In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
import os

In [16]:
class EmotionDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        wav_data = self.df.iloc[idx]["wav_file"]  
        valence = self.df.iloc[idx]["Valence"]
        arousal = self.df.iloc[idx]["Arousal"]
        dominance = self.df.iloc[idx]["Dominance"]
        
        
        inputs = self.processor(wav_data, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs['labels'] = torch.tensor([valence, arousal, dominance], dtype=torch.float32)
        
        return inputs


In [17]:
class Wav2Vec2ForEmotionRegression(nn.Module):
    def __init__(self):
        super(Wav2Vec2ForEmotionRegression, self).__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.regression_layer = nn.Linear(self.wav2vec2.config.hidden_size, 3)  # 3 for valence, arousal, dominance

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values=input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        # Take the mean across the time dimension to get a fixed-size representation
        pooled_output = hidden_states.mean(dim=1)
        return self.regression_layer(pooled_output)

In [18]:
def custom_collate(batch):
    input_values = [item['input_values'].squeeze(0) for item in batch]
    attention_mask = [item['attention_mask'].squeeze(0) if 'attention_mask' in item else torch.ones_like(item['input_values'].squeeze(0)) for item in batch]
    labels = torch.stack([item['labels'] for item in batch])

    # Pad input values and attention mask to the longest in the batch
    input_values_padded = pad_sequence(input_values, batch_first=True)
    attention_mask_padded = pad_sequence(attention_mask, batch_first=True)

    return {
        'input_values': input_values_padded,
        'attention_mask': attention_mask_padded,
        'labels': labels
    }

In [19]:
def save_checkpoint(model, optimizer, epoch, filename="model_checkpoint.pth"):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved at epoch {epoch + 1}")

def load_checkpoint(model, optimizer, filename="model_checkpoint.pth"):
    if os.path.isfile(filename):
        checkpoint = torch.load(filename, map_location="cpu", weights_only=True)  # Use "cpu" if using CPU, change if using GPU
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch
    else:
        print("No checkpoint found, starting from scratch.")
        return 0


In [20]:
def train(model, train_dataloader, test_dataloader, epochs=5):
    optimizer = AdamW(model.parameters(), lr=1e-4)
    """if torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")"""

    model.train()

    for epoch in range(epochs):
        epoch_loss = 0
        optimizer.zero_grad()

        for batch in tqdm(train_dataloader):
            input_values = batch['input_values'].to("cpu")
            attention_mask = batch['attention_mask'].to("cpu")
            labels = batch['labels'].to("cpu")
            
            optimizer.zero_grad()
            outputs = model(input_values=input_values, attention_mask=attention_mask)
            loss = nn.MSELoss()(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_loss / len(train_dataloader)}")

        save_checkpoint(model, optimizer, epoch)

        # Validation Loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in test_dataloader:
                input_values = batch['input_values'].to("cpu")
                attention_mask = batch['attention_mask'].to("cpu")
                labels = batch['labels'].to("cpu")
                
                outputs = model(input_values=input_values, attention_mask=attention_mask)
                loss = nn.MSELoss()(outputs, labels)
                val_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Validation Loss: {val_loss / len(test_dataloader)}")


In [21]:
def load_trained_model(checkpoint_path="model_checkpoint.pth"):
    model = Wav2Vec2ForEmotionRegression().to("cpu")
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(checkpoint['model_state_dict'])
        print("Loaded trained model from checkpoint.")
    else:
        print("Checkpoint not found. Using untrained model.")
    
    return model, processor

In [22]:
def predict_emotion(model, processor, wav_data):
    model.eval()
    inputs = processor(wav_data, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs['input_values'].to("cpu")
    attention_mask = inputs['attention_mask'].to("cpu") if 'attention_mask' in inputs else None

    with torch.no_grad():
        outputs = model(input_values=input_values, attention_mask=attention_mask)
    
    valence, arousal, dominance = outputs.squeeze().tolist()
    return {
        "Valence": valence,
        "Arousal": arousal,
        "Dominance": dominance
    }

In [23]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForEmotionRegression().to("cpu")

df = pd.read_pickle("./data/IEMOCAP_useful")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/configuration_utils.py:302: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [24]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
#df_sampled = df_shuffled.sample(frac=0.01, random_state=42)

In [25]:
train_df, test_df = train_test_split(df_shuffled, test_size=0.2, random_state=42)

# Create datasets
train_dataset = EmotionDataset(train_df, processor)
test_dataset = EmotionDataset(test_df, processor)

In [26]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=custom_collate)

In [27]:
train(model, train_dataloader, test_dataloader)


  0%|          | 0/2008 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
100%|██████████| 2008/2008 [3:00:46<00:00,  5.40s/it]  


Epoch 1/5, Training Loss: 0.02802917581613721
Checkpoint saved at epoch 1
Epoch 1/5, Validation Loss: 0.027061446077297942


100%|██████████| 2008/2008 [2:07:43<00:00,  3.82s/it]  


Epoch 2/5, Training Loss: 0.02687848044827903
Checkpoint saved at epoch 2


In [ ]:
#model, processor = load_trained_model("model_checkpoint.pth")

In [ ]:
"""from vocal_assistant import VocalAssistant

vc = VocalAssistant(1)
vc.talk("What is your mood today?")
while True:
    command, vocal_file = vc.take_command()
    print(command)
    break

# Predict emotions
emotions = predict_emotion(model, processor, vocal_file)
print(emotions)"""